# 06 — Prepare the operations agent: pinned SQL evidence and policy RAG

Requires the 0.6.2 project wheel and authenticated Spark/Delta. The user reports
SCORING PASS for 600 rows, 450 alerts and 150 non-alerts. This notebook independently
checks the existing data; it never loads a model, trains, scores or writes a table.

**AGENT DATA READY is not a deployed-agent or RAG acceptance result.** Live native
SQL tools, policy ingestion and agent answers must be tested separately. See
`docs/operations-agent.md`. Preserve the working notebook 05 kernel until ready
to attach/restart compute for the new wheel. No repeat of 01–05 is needed.


In [ ]:
# One configuration section. Prediction version is resolved once, then pinned.
CONFIG = {"predictions_version": None, "output_dir": None}
receipt = snapshot = sql_verification = bundle_result = None


## 1. Read-only data/provenance gate

Reads a fixed predictions version and its recorded scoring-input version.
Verifies all 600 unique IDs, customer/acceptance matching, model/run identity,
finite probabilities, saved decision consistency and homogeneous provenance.
The actual saved threshold is read, never independently configured.


In [ ]:
receipt = snapshot = sql_verification = bundle_result = None
try:
    import json
    import aidp_lakehouse
    from aidp_lakehouse.agent_queries import AgentSnapshot, query_sql
    from aidp_lakehouse.agent_readiness import prepare_agent_data
    if aidp_lakehouse.__version__ != "0.6.2":
        raise RuntimeError("Install the 0.6.2 project wheel without replacing native dependencies")
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    receipt = prepare_agent_data(spark, CONFIG["predictions_version"])
    snapshot = AgentSnapshot(receipt["predictions_version"], receipt["scoring_version"])
    print(json.dumps(receipt, indent=2))
except BaseException:
    receipt = snapshot = sql_verification = bundle_result = None
    raise


## 2. Check query semantics on Spark

These queries run in the notebook's Spark session. They do not validate the
AI compute's native SQL tool connection. No-match handling is tested separately.


In [ ]:
sql_verification = bundle_result = None
try:
    if receipt is None or snapshot is None:
        raise RuntimeError("Complete the data/provenance gate first")
    summary = spark.sql(query_sql("risk_summary", {}, snapshot)).first().asDict()
    if summary["open_orders"] != receipt["rows"] or summary["alert_orders"] != receipt["alerts"]:
        raise ValueError("SQL summary does not reconcile")
    top = spark.sql(query_sql("rank_open_orders", {"limit": 10}, snapshot)).collect()
    if len(top) != 10 or [r["order_id"] for r in top] != [r["order_id"] for r in
            sorted(top, key=lambda r: (-r["late_probability"], r["order_id"]))]:
        raise ValueError("Risk ranking is not deterministic")
    found = spark.sql(query_sql("get_order_risk", {"order_id": top[0]["order_id"]}, snapshot)).collect()
    if len(found) != 1 or found[0]["late_probability"] != top[0]["late_probability"]:
        raise ValueError("Order lookup differs from ranking")
    if spark.sql(query_sql("get_order_risk", {"order_id": "O-99999999"}, snapshot)).count() != 0:
        raise ValueError("Missing-order check failed")
    for tool in ("risk_by_carrier", "risk_by_region"):
        groups = spark.sql(query_sql(tool, {}, snapshot)).collect()
        if sum(r["open_orders"] for r in groups) != receipt["rows"] or sum(r["alert_orders"] for r in groups) != receipt["alerts"]:
            raise ValueError("Grouped counts do not reconcile")
    customer = spark.sql(query_sql("get_customer_orders", {"customer_id": top[0]["customer_id"]}, snapshot)).collect()
    if not customer or any(r["customer_id"] != top[0]["customer_id"] for r in customer):
        raise ValueError("Customer filter failed")
    sql_verification = {"status": "NOTEBOOK SQL PASS", "native_agent_sql_verified": False}
    print(json.dumps({"summary": summary, "sql_verification": sql_verification}, indent=2, default=str))
    spark.sql(query_sql("rank_open_orders", {"limit": 10}, snapshot)).show(truncate=False)
except BaseException:
    sql_verification = bundle_result = None
    raise


## 3. Optional local handoff bundle

This preserves the original Custom Code ZIP and the two byte-preserved TXT policies.
The Custom Code deployment had a runtime-loader failure; it is not the active agent.
For the verified native SQL alternative, prepare definitions locally with
`python -m aidp_lakehouse.agent_native_sql --profile complete`, the existing readiness
receipt and the private verified routing configuration. See `docs/operations-agent.md`.
Generated deployment configuration stays outside Git. Never ingest this bundle:
only the two TXT in `documents/` belong in the dedicated policy source.


In [ ]:
bundle_result = None
if receipt is None or sql_verification is None:
    raise RuntimeError("Complete data and notebook SQL checks first")
if CONFIG["output_dir"] is None:
    print("Bundle not written. Save the receipt and use the local bundle CLI, or set output_dir.")
else:
    from aidp_lakehouse.agent_bundle import write_agent_bundle
    bundle_result = write_agent_bundle(CONFIG["output_dir"], receipt)
    print(json.dumps(bundle_result, indent=2))


## 4. Native tools, RAG and acceptance

Reuse the dedicated DEV agent and existing compute; verify their identities first.
The current route uses six bounded native Spark SQL tools and native RAG, without
Custom Code. The corrected `operations_policies` KB uses the dedicated `/documents`
source only. Consult `docs/operations-agent-execution.md` for actual ingestion,
retrieval and acceptance results; notebook readiness is not agent acceptance.

No repeat of notebooks 01–05, training or scoring is required. Do not create,
resize, restart compute or change inactivity policies from this notebook.


In [ ]:
if receipt is None or sql_verification is None or sql_verification.get("status") != "NOTEBOOK SQL PASS":
    raise RuntimeError("AGENT PREPARATION FAIL: data or notebook SQL checks missing")
print("AGENT DATA READY: 600 pinned predictions reconciled; notebook SQL verified. Native tools, RAG and agent acceptance pending.")
